# ✅ Agent 3: Document Validation Agent

**Purpose:** Validates extracted data against business rules and compliance checks.

| Validation Type | Rules |
|----------------|-------|
| 📄 Invoice | Total matches line items, valid dates, tax calculations, payment terms present |
| 📋 Contract | Required clauses exist, dates valid, parties named, term reasonable |
| 📊 Report | Metrics present, period specified, recommendations actionable |
| 📝 Email | Sender/recipient valid, subject present, date parseable |

**Tech:** Gemini 2.0 Flash via Vertex AI + Python validation logic

In [ ]:
from google import genai
from google.genai import types
import json
from datetime import datetime

# Initialize Vertex AI client
client = genai.Client(
    vertexai=True,
    project="pwc-agentic-demo",
    location="us-central1"
)

print("✅ Vertex AI client ready for validation!")

## 📏 Define Validation Rules per Document Type

In [ ]:
VALIDATION_PROMPTS = {
    "Invoice": """You are an invoice validation expert. Validate the following extracted invoice data.

Check these rules:
1. INTEGRITY: Do line item amounts sum to subtotal?
2. TAX: Is tax calculation correct (subtotal × tax rate = tax amount)?
3. TOTAL: Does subtotal + tax = total?
4. DATES: Is the invoice date valid and not in the future?
5. FIELDS: Are all required fields present (invoice number, date, vendor, total)?
6. AMOUNTS: Are all amounts positive and in consistent currency?
7. PAYMENT TERMS: Are payment terms specified and reasonable (Net 15-90)?

Return as JSON:
{{
  "is_valid": true/false,
  "score": 0.0-1.0,
  "checks": [
    {{"rule": "rule name", "status": "pass/fail/warning", "details": "explanation"}}
  ],
  "issues": ["list of critical issues"],
  "warnings": ["list of non-critical warnings"]
}}

Extracted Data:
{data}""",

    "Contract": """You are a contract validation expert. Validate the following extracted contract data.

Check these rules:
1. PARTIES: Are both parties clearly identified with names?
2. DATES: Is effective date valid? Is term duration specified?
3. COMPENSATION: Is compensation amount specified and reasonable?
4. SCOPE: Is the scope of work clearly defined?
5. TERMINATION: Is a termination clause present?
6. CONFIDENTIALITY: Is a confidentiality clause present?
7. TERM: Is the contract term reasonable (1-36 months)?

Return as JSON:
{{
  "is_valid": true/false,
  "score": 0.0-1.0,
  "checks": [
    {{"rule": "rule name", "status": "pass/fail/warning", "details": "explanation"}}
  ],
  "issues": ["list of critical issues"],
  "warnings": ["list of non-critical warnings"]
}}

Extracted Data:
{data}""",

    "Report": """You are a report validation expert. Validate the following extracted report data.

Check these rules:
1. TITLE: Is the report title present and descriptive?
2. PERIOD: Is the reporting period clearly specified?
3. SUMMARY: Is the executive summary present and meaningful (>50 chars)?
4. METRICS: Are at least 3 key metrics present with values?
5. TRENDS: Do metrics include trend indicators (up/down/stable)?
6. CHALLENGES: Are challenges documented?
7. RECOMMENDATIONS: Are actionable recommendations provided?

Return as JSON:
{{
  "is_valid": true/false,
  "score": 0.0-1.0,
  "checks": [
    {{"rule": "rule name", "status": "pass/fail/warning", "details": "explanation"}}
  ],
  "issues": ["list of critical issues"],
  "warnings": ["list of non-critical warnings"]
}}

Extracted Data:
{data}""",

    "Email": """You are an email validation expert. Validate the following extracted email data.

Check these rules:
1. SENDER: Is the sender field present and contains a valid name/email?
2. RECIPIENT: Is the recipient field present?
3. SUBJECT: Is the subject line present and meaningful?
4. DATE: Is the date present and parseable?
5. SUMMARY: Is the body summary present and coherent?
6. SENTIMENT: Is sentiment analysis present and reasonable?

Return as JSON:
{{
  "is_valid": true/false,
  "score": 0.0-1.0,
  "checks": [
    {{"rule": "rule name", "status": "pass/fail/warning", "details": "explanation"}}
  ],
  "issues": ["list of critical issues"],
  "warnings": ["list of non-critical warnings"]
}}

Extracted Data:
{data}"""
}

print(f"✅ Validation prompts loaded for: {list(VALIDATION_PROMPTS.keys())}")

## 🔧 Define the Validation Function

In [ ]:
def validate_document(extracted_data: dict, doc_type: str) -> dict:
    """
    Validates extracted document data against business rules.
    
    Args:
        extracted_data: The extracted fields from Agent 2
        doc_type: One of 'Invoice', 'Contract', 'Report', 'Email'
    
    Returns:
        dict: Validation results with pass/fail status, score, and detailed checks
    """
    if doc_type not in VALIDATION_PROMPTS:
        return {
            "is_valid": False,
            "score": 0.0,
            "error": f"Unknown document type: {doc_type}",
            "checks": [],
            "issues": [f"Cannot validate unknown document type: {doc_type}"],
            "warnings": []
        }
    
    data_str = json.dumps(extracted_data, indent=2)
    
    prompt = VALIDATION_PROMPTS[doc_type].format(data=data_str)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )

    return json.loads(response.text)

## 🧪 Test 1: Validate an Invoice (Should Pass)

In [ ]:
valid_invoice_data = {
    "invoice_number": "INV-2024-0042",
    "date": "2024-03-15",
    "vendor": {
        "name": "TechCorp Solutions Pvt. Ltd.",
        "address": "123 Innovation Hub, HITEC City, Hyderabad"
    },
    "bill_to": {
        "name": "PricewaterhouseCoopers",
        "address": "5th Floor, Block B, Cyber Towers, Hyderabad"
    },
    "line_items": [
        {"description": "AI Consulting Services", "quantity": 40, "rate": "$150/hr", "amount": "$6,000.00"},
        {"description": "Cloud Infrastructure Setup", "quantity": 1, "rate": "$3,500.00", "amount": "$3,500.00"}
    ],
    "subtotal": "$9,500.00",
    "tax": "$1,710.00",
    "total": "$11,210.00",
    "payment_terms": "Net 30 days",
    "bank_details": "HDFC Bank, A/C: 50100012345678, IFSC: HDFC0001234"
}

result = validate_document(valid_invoice_data, "Invoice")
print(f"{'✅' if result['is_valid'] else '❌'} Valid: {result['is_valid']}")
print(f"📊 Score: {result['score']}")
print(f"\n📋 Checks:")
for check in result['checks']:
    icon = "✅" if check['status'] == 'pass' else "⚠️" if check['status'] == 'warning' else "❌"
    print(f"  {icon} {check['rule']}: {check['details']}")
if result.get('issues'):
    print(f"\n🚨 Issues: {result['issues']}")
if result.get('warnings'):
    print(f"\n⚠️ Warnings: {result['warnings']}")

## 🧪 Test 2: Validate an Invoice (Should Fail — Bad Data)

In [ ]:
bad_invoice_data = {
    "invoice_number": None,
    "date": "2027-12-31",
    "vendor": {
        "name": "Some Corp",
        "address": None
    },
    "bill_to": {
        "name": "Client",
        "address": None
    },
    "line_items": [
        {"description": "Service A", "quantity": 10, "rate": "$100", "amount": "$2,000.00"}
    ],
    "subtotal": "$1,000.00",
    "tax": "$180.00",
    "total": "$1,500.00",
    "payment_terms": None,
    "bank_details": None
}

result = validate_document(bad_invoice_data, "Invoice")
print(f"{'✅' if result['is_valid'] else '❌'} Valid: {result['is_valid']}")
print(f"📊 Score: {result['score']}")
print(f"\n📋 Checks:")
for check in result['checks']:
    icon = "✅" if check['status'] == 'pass' else "⚠️" if check['status'] == 'warning' else "❌"
    print(f"  {icon} {check['rule']}: {check['details']}")
if result.get('issues'):
    print(f"\n🚨 Issues: {result['issues']}")
if result.get('warnings'):
    print(f"\n⚠️ Warnings: {result['warnings']}")

## 🧪 Test 3: Validate a Contract

In [ ]:
contract_data = {
    "contract_title": "Master Services Agreement",
    "effective_date": "2024-01-01",
    "parties": [
        {"name": "ABC Technology Pvt. Ltd.", "role": "Provider"},
        {"name": "XYZ Consulting", "role": "Client"}
    ],
    "scope": "AI-powered document processing solutions including classification, extraction, and validation.",
    "term": "12 months",
    "compensation": "$150,000 annually, payable quarterly",
    "key_terms": [
        {"clause": "Confidentiality", "summary": "Both parties maintain strict confidentiality for 2 years post-termination."},
        {"clause": "IP Rights", "summary": "Provider retains IP rights to pre-existing technology."}
    ],
    "termination_clause": "Either party may terminate with 30 days written notice. Immediate termination for non-payment exceeding 15 days."
}

result = validate_document(contract_data, "Contract")
print(f"{'✅' if result['is_valid'] else '❌'} Valid: {result['is_valid']}")
print(f"📊 Score: {result['score']}")
print(f"\n📋 Checks:")
for check in result['checks']:
    icon = "✅" if check['status'] == 'pass' else "⚠️" if check['status'] == 'warning' else "❌"
    print(f"  {icon} {check['rule']}: {check['details']}")
if result.get('issues'):
    print(f"\n🚨 Issues: {result['issues']}")
if result.get('warnings'):
    print(f"\n⚠️ Warnings: {result['warnings']}")

## 🧪 Test 4: Validate a Report

In [ ]:
report_data = {
    "title": "Quarterly Performance Report - Q4 2023",
    "period": "October 1 - December 31, 2023",
    "executive_summary": "Revenue increased by 23% YoY to $4.2M with improved client acquisition.",
    "key_metrics": [
        {"metric": "Total Revenue", "value": "$4,200,000", "trend": "up"},
        {"metric": "New Clients", "value": "12", "trend": "up"},
        {"metric": "Client Satisfaction", "value": "4.6/5.0", "trend": "stable"}
    ],
    "challenges": ["Cloud costs exceeded budget by 8%", "Two senior engineers resigned"],
    "recommendations": ["Negotiate better cloud pricing", "Implement retention bonuses", "Expand sales team"]
}

result = validate_document(report_data, "Report")
print(f"{'✅' if result['is_valid'] else '❌'} Valid: {result['is_valid']}")
print(f"📊 Score: {result['score']}")
print(f"\n📋 Checks:")
for check in result['checks']:
    icon = "✅" if check['status'] == 'pass' else "⚠️" if check['status'] == 'warning' else "❌"
    print(f"  {icon} {check['rule']}: {check['details']}")
if result.get('issues'):
    print(f"\n🚨 Issues: {result['issues']}")
if result.get('warnings'):
    print(f"\n⚠️ Warnings: {result['warnings']}")

## 🔗 Full Pipeline Test: Classify → Extract → Validate

In [ ]:
def classify_document(text: str) -> dict:
    """Agent 1: Classify"""
    prompt = f"""You are a document classification expert.
Classify the following document into exactly ONE category:
- Invoice
- Contract
- Report
- Email
- Unknown

Respond in this exact JSON format:
{{"type": "<category>", "confidence": <0.0-1.0>, "reasoning": "<brief explanation>"}}

Document:
---
{text}
---"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )
    return json.loads(response.text)


def extract_fields(text: str, doc_type: str) -> dict:
    """Agent 2: Extract"""
    from .02_extraction_agent import EXTRACTION_PROMPTS
    # Inline for testing
    prompts = {
        "Invoice": "Extract invoice fields as JSON: invoice_number, date, vendor, bill_to, line_items, subtotal, tax, total, payment_terms, bank_details",
        "Contract": "Extract contract fields as JSON: contract_title, effective_date, parties, scope, term, compensation, key_terms, termination_clause",
        "Report": "Extract report fields as JSON: title, period, executive_summary, key_metrics, challenges, recommendations",
        "Email": "Extract email fields as JSON: from, to, cc, subject, date, body_summary, action_items, sentiment"
    }
    prompt = f"Extract structured data from this document. Return ONLY JSON.\n\nDocument:\n{text}"
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )
    return json.loads(response.text)


def process_document_full(text: str) -> dict:
    """
    Full 3-agent pipeline: Classify → Extract → Validate
    """
    print("\n🔄 Starting 3-Agent Pipeline...")
    
    # Agent 1: Classify
    print("\n📄 Agent 1: Classifying document...")
    classification = classify_document(text)
    doc_type = classification["type"]
    confidence = classification["confidence"]
    print(f"   Type: {doc_type} (confidence: {confidence})")
    
    # Agent 2: Extract
    print(f"\n🔍 Agent 2: Extracting fields for {doc_type}...")
    if doc_type != "Unknown":
        extracted = extract_fields(text, doc_type)
        print(f"   Extracted {len(extracted)} fields")
    else:
        extracted = {"message": "Cannot extract from unknown document type"}
        print("   ⚠️ Skipped (unknown type)")
    
    # Agent 3: Validate
    print(f"\n✅ Agent 3: Validating extracted data...")
    if doc_type != "Unknown":
        validation = validate_document(extracted, doc_type)
        status = "✅ PASSED" if validation["is_valid"] else "❌ FAILED"
        print(f"   {status} (score: {validation['score']})")
    else:
        validation = {"is_valid": False, "score": 0.0}
        print("   ⚠️ Skipped (unknown type)")
    
    return {
        "classification": classification,
        "extracted_fields": extracted,
        "validation": validation
    }

In [ ]:
# Test full pipeline
sample = """
INVOICE #INV-2024-0042
Date: March 15, 2024
From: TechCorp Solutions Pvt. Ltd.
To: PricewaterhouseCoopers

AI Consulting Services - 40 hrs @ $150/hr = $6,000
Cloud Infrastructure Setup - 1 unit @ $3,500 = $3,500

Subtotal: $9,500
Tax (18%): $1,710
Total: $11,210

Payment Terms: Net 30 days
Bank: HDFC Bank, A/C: 50100012345678
"""

result = process_document_full(sample)
print("\n\n📋 Full Pipeline Result:")
print(json.dumps(result, indent=2, default=str))

## ✅ Validation Agent - Complete

**What we built:**
- Type-specific validation rules for Invoice, Contract, Report, Email
- `validate_document()` with scoring, pass/fail, and detailed checks
- Full 3-agent pipeline: Classify → Extract → Validate

**Next:** Summary Agent (Agent 4) — generates executive summary with insights